# 02 · Trading calendars, alignment, returns and rolling statistics

**Goal:** put every series on the **Indian FX trading calendar**, compute returns, and add 20-day volatility and
60-day correlation.

> **Series:** 01 Ingest → 02 Align & transform → 03 Event windows → 04 Database → 05 Results report → 06 Analysis & charts. Run them in order (or run `python run_pipeline.py` from the project folder). Paths and settings live in `config.py`.

**Two calendars, never mixed.**
- **RBI** (primary): every date with an RBI/FBIL reference rate.
- **FRED** (cross-check): every date with a Fed H.10 USD/INR rate, which uses the New York clock.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

import numpy as np
import pandas as pd
from config import (PROC, ANALYSIS_START, ANALYSIS_END, MAX_STALENESS_DAYS, VOL_WINDOW,
                    VOL_MIN_OBS, CORR_WINDOW, CORR_MIN_OBS, ANNUALISE, PANEL_SERIES,
                    CORR_PAIRS, FX_SOURCES, LEVEL_CHANGE_SERIES)

STEP_SERIES = {"FFR_UPPER", "IN_REPO"}

In [2]:
def load_obs():
    o = pd.read_csv(PROC / "observations_long.csv", parse_dates=["obs_date"])
    return {k: g.set_index("obs_date")["value"].sort_index() for k, g in o.groupby("series_id")}

obs = load_obs()
print(len(obs), "series loaded")

24 series loaded


## Aligning different frequencies: the as-of join

For every Indian trading date, take the **latest value on or before that date**, but never one older than
**5 calendar days**, and **never a later value** (no look-ahead). Policy rates are step series: a rate stays in force until
it changes, so they carry forward with no limit. `staleness_days` records how old each value was.

In [3]:
def asof_align(calendar, s, step=False):
    """Align series s to calendar dates. Returns value, source obs date, staleness (days)."""
    left = pd.DataFrame({"date": calendar})
    right = pd.DataFrame({"obs_date": s.index, "value": s.values})
    tol = None if step else pd.Timedelta(days=MAX_STALENESS_DAYS)
    m = pd.merge_asof(left, right, left_on="date", right_on="obs_date",
                      direction="backward", tolerance=tol)
    m["staleness_days"] = (m["date"] - m["obs_date"]).dt.days
    return m

In [4]:
# example: Brent aligned to the first RBI trading days of 2020 (1 Jan was a holiday for Brent)
cal = obs["USDINR_RBI"].loc["2019-12-30":"2020-01-08"].index
asof_align(cal, obs["BRENT"])

,date,obs_date,value,staleness_days
0,2019-12-30,2019-12-30,68.30,0
1,2019-12-31,2019-12-31,67.77,0
2,2020-01-01,2019-12-31,67.77,1
3,2020-01-02,2020-01-02,67.05,0
4,2020-01-03,2020-01-03,69.08,0
5,2020-01-06,2020-01-06,70.25,0
6,2020-01-07,2020-01-07,68.74,0
7,2020-01-08,2020-01-08,67.31,0


## Returns

- **Prices** use log returns, `ln(Pₜ / Pₜ₋₁)`, because they add up over time. This is what volatility and correlation use.
- **Rates and yields** use changes in **percentage points**, never "% of a yield".

In [5]:
def change(series_id, v):
    """Daily change: log return for prices, first difference for rates/yields."""
    if series_id in LEVEL_CHANGE_SERIES:
        return v.diff()
    return np.log(v / v.shift(1))

## Building the daily panel

For each calendar: align every series, compute USD/INR returns in both directions (USD/INR % and the rupee's own %), then
**20-day annualised volatility** and **60-day rolling correlation** of USD/INR with each market.

In [6]:
def build_panel(obs, basis):
    fx_id = FX_SOURCES[basis]
    fx = obs[fx_id]
    cal = fx[(fx.index >= pd.Timestamp(ANALYSIS_START) - pd.DateOffset(months=6))
             & (fx.index <= pd.Timestamp(ANALYSIS_END))].index
    wide = pd.DataFrame(index=cal)
    wide.index.name = "date"
    wide["USDINR"] = fx.reindex(cal).values
    long_rows, dq = [], []
    for sid in PANEL_SERIES:
        if sid not in obs or obs[sid].empty:
            wide[sid] = np.nan
            dq.append(dict(basis=basis, series_id=sid, days=len(cal), exact=0, carried=0,
                           missing=len(cal), max_staleness=None, note="no source data"))
            continue
        m = asof_align(cal, obs[sid], step=sid in STEP_SERIES)
        wide[sid] = m["value"].values
        st = m["staleness_days"]
        # series that start after the calendar starts are 'not yet available', not missing
        first = obs[sid].index.min()
        live = cal >= first
        dq.append(dict(basis=basis, series_id=sid, days=int(live.sum()),
                       exact=int(((st == 0) & live).sum()),
                       carried=int(((st > 0) & live).sum()),
                       missing=int((m["value"].isna() & live).sum()),
                       max_staleness=None if st.isna().all() else int(st[live].max()),
                       note="step series (no staleness cap)" if sid in STEP_SERIES else
                       f"as-of join, cap {MAX_STALENESS_DAYS} calendar days"))
        long_rows.append(pd.DataFrame({"basis": basis, "date": cal, "series_id": sid,
                                       "value": m["value"].values,
                                       "source_obs_date": m["obs_date"].values,
                                       "staleness_days": st.values}))
    # returns & rolling stats
    wide["USDINR_ret"] = np.log(wide["USDINR"] / wide["USDINR"].shift(1))
    wide["INR_pct"] = (wide["USDINR"].shift(1) / wide["USDINR"] - 1) * 100   # rupee's own % change
    wide["USDINR_pct"] = (wide["USDINR"] / wide["USDINR"].shift(1) - 1) * 100
    for sid in PANEL_SERIES:
        wide[f"{sid}_ret"] = change(sid, wide[sid])
    for sid in ["USDINR"] + [s for s in PANEL_SERIES if s not in LEVEL_CHANGE_SERIES]:
        r = wide[f"{sid}_ret"]
        wide[f"{sid}_vol20"] = r.rolling(VOL_WINDOW, min_periods=VOL_MIN_OBS).std() * np.sqrt(ANNUALISE) * 100
    for sid in CORR_PAIRS:
        wide[f"corr60_USDINR_{sid}"] = wide["USDINR_ret"].rolling(
            CORR_WINDOW, min_periods=CORR_MIN_OBS).corr(wide[f"{sid}_ret"])
    wide = wide[wide.index >= pd.Timestamp(ANALYSIS_START)]
    wide["basis"] = basis
    wide["trading_day_seq"] = np.arange(1, len(wide) + 1)
    fxlong = pd.DataFrame({"basis": basis, "date": wide.index, "series_id": fx_id,
                           "value": wide["USDINR"].values, "source_obs_date": wide.index,
                           "staleness_days": 0})
    long = pd.concat([fxlong] + long_rows)
    long = long[long["date"] >= pd.Timestamp(ANALYSIS_START)]
    return wide, long, dq

In [7]:
panels, longs, dqs = {}, [], []
for basis in FX_SOURCES:
    wide, long, dq = build_panel(obs, basis)
    wide.reset_index().to_csv(PROC / f"panel_wide_{basis}.csv", index=False, date_format="%Y-%m-%d")
    panels[basis] = wide; longs.append(long); dqs += dq
    print(f"{basis:5s} calendar: {wide.index.min():%d %b %Y} to {wide.index.max():%d %b %Y}, {len(wide):,} trading days")
pd.concat(longs).to_csv(PROC / "panel_daily_long.csv", index=False, date_format="%Y-%m-%d")

panels["RBI"][["USDINR", "USDINR_pct", "INR_pct", "BRENT", "DXY_BROAD", "USDINR_vol20", "corr60_USDINR_BRENT"]].tail()

RBI   calendar: 01 Jan 2013 to 29 Sep 2026, 3,315 trading days


FRED  calendar: 02 Jan 2013 to 25 Sep 2026, 3,434 trading days


,USDINR,USDINR_pct,INR_pct,BRENT,DXY_BROAD,USDINR_vol20,corr60_USDINR_BRENT
date,,,,,,,
2026-09-23,95.7310,-0.090693,0.090775,114.89,120.1169,4.241199,0.078314
2026-09-24,95.9099,0.186878,-0.186529,114.89,120.5521,4.291919,0.086395
2026-09-25,95.8918,-0.018872,0.018875,114.89,120.3300,4.214473,0.092806
2026-09-28,95.9681,0.079569,-0.079506,NaN,120.3300,4.218533,0.117354
2026-09-29,96.0321,0.066689,-0.066644,NaN,120.3300,4.189588,0.116870


## How well did the other series line up with the Indian calendar?

In [8]:
alignment = pd.DataFrame(dqs)
alignment["pct_exact"] = (alignment.exact / alignment.days * 100).round(1)
alignment.pivot(index="series_id", columns="basis", values="pct_exact")

basis,FRED,RBI
series_id,,
BRENT,98.5,97.6
DXY_BROAD,99.6,95.6
FFR_UPPER,100.0,100.0
GOLD_USD,97.9,97.5
IN_REPO,0.9,1.0
NIFTY50,94.7,100.0
UST10Y,99.7,96.2
VIX,99.6,97.8


In [9]:
alignment.to_csv(PROC / "dq_alignment.csv", index=False)

## Date dimension

In [10]:
def dim_date(obs):
    d = pd.DataFrame({"date": pd.date_range(ANALYSIS_START, ANALYSIS_END, freq="D")})
    d["date_key"] = d["date"].dt.strftime("%Y%m%d").astype(int)
    d["year"] = d["date"].dt.year
    d["quarter"] = "Q" + d["date"].dt.quarter.astype(str)
    d["month_num"] = d["date"].dt.month
    d["month_name"] = d["date"].dt.strftime("%b")
    d["year_month"] = d["date"].dt.strftime("%Y-%m")
    d["day_of_week"] = d["date"].dt.strftime("%a")
    d["is_weekend"] = (d["date"].dt.dayofweek >= 5).astype(int)
    # Indian financial year (Apr-Mar), useful for RBI-context slicing
    d["fiscal_year_in"] = np.where(d["month_num"] >= 4, d["year"], d["year"] - 1)
    d["fiscal_year_in"] = "FY" + (d["fiscal_year_in"] + 1).astype(str).str[-2:]
    rbi_days = set(obs["USDINR_RBI"].index)
    fred_days = set(obs["USDINR_FRED"].index)
    d["is_rbi_fx_day"] = d["date"].isin(rbi_days).astype(int)
    d["is_us_h10_day"] = d["date"].isin(fred_days).astype(int)
    hol = pd.read_csv(PROC / "rbi_holiday_rows.csv", parse_dates=["obs_date"])["obs_date"]
    d["rbi_listed_holiday"] = d["date"].isin(set(hol)).astype(int)
    return d

dd = dim_date(obs)
dd.to_csv(PROC / "dim_date.csv", index=False, date_format="%Y-%m-%d")
dd.head(3)

,date,date_key,year,quarter,month_num,month_name,year_month,day_of_week,is_weekend,fiscal_year_in,is_rbi_fx_day,is_us_h10_day,rbi_listed_holiday
0,2013-01-01,20130101,2013,Q1,1,Jan,2013-01,Tue,0,FY13,1,0,0
1,2013-01-02,20130102,2013,Q1,1,Jan,2013-01,Wed,0,FY13,1,1,0
2,2013-01-03,20130103,2013,Q1,1,Jan,2013-01,Thu,0,FY13,1,1,0


## Cross-check: RBI reference rate vs the Fed's H.10 rate

The two rates are fixed about **9 hours apart** (~13:30 IST vs noon in New York). Levels agree closely, but daily
returns correlate only moderately. Weekly returns wash most of the timing noise out.

In [11]:
def fx_crosscheck(obs):
    a = obs["USDINR_RBI"].rename("rbi_ref")
    b = obs["USDINR_FRED"].rename("fred_h10")
    x = pd.concat([a, b], axis=1, sort=True)
    x = x[(x.index >= pd.Timestamp(ANALYSIS_START))]
    x["status"] = np.select([x.rbi_ref.notna() & x.fred_h10.notna(), x.rbi_ref.notna()],
                            ["both", "rbi_only"], "fred_only")
    x["diff_inr"] = x["fred_h10"] - x["rbi_ref"]
    x["diff_pct"] = x["diff_inr"] / x["rbi_ref"] * 100
    x["large_gap_flag"] = (x["diff_pct"].abs() > 0.5).astype(int)
    x.index.name = "date"
    return x.reset_index()

xc = fx_crosscheck(obs)
xc.to_csv(PROC / "fx_crosscheck.csv", index=False, date_format="%Y-%m-%d")
both = xc[xc.status == "both"].set_index("date")
daily = np.log(both[["rbi_ref", "fred_h10"]]).diff().corr().iloc[0, 1]
weekly = np.log(both[["rbi_ref", "fred_h10"]].resample("W-FRI").last()).diff().corr().iloc[0, 1]
print(f"common days: {len(both):,} | mean |gap|: {both.diff_pct.abs().mean():.3f}% | "
      f"daily-return corr: {daily:.2f} | weekly-return corr: {weekly:.2f}")

common days: 3,180 | mean |gap|: 0.174% | daily-return corr: 0.56 | weekly-return corr: 0.89


## Monthly context: CPI, reserves, RBI intervention, policy rates

Monthly data stays monthly; nothing is interpolated to daily. US CPI YoY is computed on a complete monthly grid (the
US government shutdown left October 2025 unpublished). India CPI uses MoSPI's **published** YoY.

In [12]:
def macro_monthly(obs):
    rows = []
    for sid in ["US_CPI", "IN_CPI_OECD", "IN_FX_RESERVES", "IN_RBI_FX_NET", "IN_RBI_FX_PURCHASE",
                "IN_RBI_FX_SALE", "IN_RBI_FWD_BOOK"]:
        if sid not in obs:
            continue
        s = obs[sid].copy()
        s.index = s.index.to_period("M").to_timestamp()
        # reindex to a complete monthly grid so shift(12) is always 12 months
        # (e.g. US CPI for Oct-2025 was never published)
        s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="MS"))
        df = pd.DataFrame({"month": s.index, "series_id": sid, "value": s.values})
        is_index = sid in ("US_CPI", "IN_CPI_OECD")
        df["yoy_pct"] = (s / s.shift(12) - 1).values * 100 if is_index else np.nan
        df["mom_change"] = (s - s.shift(1)).values if sid == "IN_FX_RESERVES" else np.nan
        df["series_version"] = ""
        rows.append(df.dropna(subset=["value"]))
    # India CPI (MoSPI): YoY exactly as PUBLISHED (not recomputed across base changes)
    cpi_path = PROC / "cpi_mospi.csv"
    if cpi_path.exists():
        c = pd.read_csv(cpi_path, parse_dates=["month"])
        rows.append(pd.DataFrame({"month": c["month"], "series_id": "IN_CPI_MOSPI", "value": c["index"],
                                  "yoy_pct": c["yoy"], "mom_change": np.nan,
                                  "series_version": "base " + c["base"].astype(str) + "=100 (" + c["status"] + ")"}))
    # policy rates as month-end values (step series)
    months = pd.date_range(ANALYSIS_START, ANALYSIS_END, freq="MS")
    for sid in ["IN_REPO", "FFR_UPPER"]:
        m = asof_align(months + pd.offsets.MonthEnd(0), obs[sid], step=True)
        rows.append(pd.DataFrame({"month": months, "series_id": sid, "value": m["value"].values,
                                  "yoy_pct": np.nan, "mom_change": np.nan, "series_version": ""}))
    out = pd.concat(rows)
    out = out[out["month"] >= pd.Timestamp(ANALYSIS_START)]
    # India-US policy-rate differential (percentage points)
    p = out.pivot_table(index="month", columns="series_id", values="value")
    diff = pd.DataFrame({"month": p.index, "series_id": "RATE_DIFF_IN_US",
                         "value": (p["IN_REPO"] - p["FFR_UPPER"]).values,
                         "yoy_pct": np.nan, "mom_change": np.nan, "series_version": ""})
    return pd.concat([out, diff])

mm = macro_monthly(obs)
mm.to_csv(PROC / "macro_monthly.csv", index=False, date_format="%Y-%m-%d")
mm[mm.series_id.isin(["IN_CPI_MOSPI", "US_CPI", "RATE_DIFF_IN_US"])].pivot_table(
    index="month", columns="series_id", values=["yoy_pct", "value"]).tail(6)

value                               yoy_pct          
series_id  IN_CPI_MOSPI RATE_DIFF_IN_US   US_CPI IN_CPI_MOSPI    US_CPI
month                                                                  
2026-04-01       105.12            1.50  332.407         3.48  3.779246
2026-05-01       105.91            1.50  333.979         3.94  4.166615
2026-06-01       107.00            1.50  332.568         4.38  3.463531
2026-07-01       107.95            1.50  332.813         4.45  3.303856
2026-08-01       108.74            1.50  334.131         4.82  3.353016
2026-09-01          NaN            1.25      NaN          NaN       NaN

**Output:** daily panels, alignment and cross-check tables in `01_Data/3_Intermediate/`. Next: **03 · Event windows**.